# Enterprise Financial Risk Intelligence & Fraud Forensics
## Notebook 12: Multi-Model Ensemble Stacking & Cost-Optimal Thresholding Engine

---

### Scientific Problem Formulation & Asymmetric Optimization Mathematics:

In high-volume transaction authorization systems, no single learning algorithm dominates across all fraud topologies. Linear models provide well-calibrated probabilities, gradient-boosted decision trees capture complex non-linear feature interactions, and extremely randomized ensembles minimize variance in dense localized clusters.

To construct a robust production defense system, we formulate a two-tier **Out-of-Fold (OOF) Stacking Ensemble** with dynamic **Cost-Optimal Thresholding**.

#### 1. Out-of-Fold (OOF) Prediction Matrix Formulation:
Given training partition $\mathcal{D}_{\text{train}} = \{(\mathbf{x}_i, y_i)\}_{i=1}^N$ partitioned into $K$ stratified folds $\{\mathcal{F}_1, \dots, \mathcal{F}_K\}$ and $M$ diverse Level-0 base learners $\{\mathcal{M}_1, \dots, \mathcal{M}_M\}$:

For each fold $k \in \{1, \dots, K\}$ and each base model $m \in \{1, \dots, M\}$:
1. Train $\mathcal{M}_m^{(k)}$ on $\mathcal{D}_{\text{train}} \setminus \mathcal{F}_k$.
2. Generate out-of-fold probability predictions on the held-out fold:
$$\hat{z}_{i, m} = P(y_i = 1 \mid \mathbf{x}_i; \mathcal{M}_m^{(k)}), \quad \forall i \in \mathcal{F}_k$$

The resulting Level-1 meta-training feature matrix $\mathbf{Z}_{\text{OOF}} \in \mathbb{R}^{N \times M}$ captures the cross-validated probabilistic outputs without target leakage:
$$\mathbf{Z}_{\text{OOF}} = \begin{bmatrix} \hat{z}_{1,1} & \hat{z}_{1,2} & \dots & \hat{z}_{1,M} \\ \hat{z}_{2,1} & \hat{z}_{2,2} & \dots & \hat{z}_{2,M} \\ \vdots & \vdots & \ddots & \vdots \\ \hat{z}_{N,1} & \hat{z}_{N,2} & \dots & \hat{z}_{N,M} \end{bmatrix}$$

For out-of-time test instances $\mathbf{x}_j \in \mathcal{D}_{\text{test}}$, meta-features are generated via fold averaging:
$$\mathbf{Z}_{\text{test}}[j, m] = \frac{1}{K} \sum_{k=1}^K P(y_j = 1 \mid \mathbf{x}_j; \mathcal{M}_m^{(k)})$$

#### 2. Level-1 Meta-Learner Optimization:
The meta-learner $g(\mathbf{z})$ maps the ensemble probability vector $\mathbf{z} \in [0, 1]^M$ to a final calibrated risk score:
$$\hat{p}_{\text{ensemble}}(\mathbf{x}) = \sigma\left(\mathbf{w}^T \mathbf{z} + b\right) = \frac{1}{1 + \exp\left(-(\sum_{m=1}^M w_m z_m + b)\right)}$$
subject to regularized log-loss minimization with optional non-negativity constraints ($w_m \ge 0$).

#### 3. Dollar-Loss Cost Matrix & Optimal Threshold Search:
The financial risk surface is asymmetric: False Negatives (missed fraud) incur transaction chargeback and direct asset loss, while False Positives (customer insult) incur operational review costs and friction-induced customer churn.

$$\text{Cost}(\theta) = \sum_{i: y_i=1, \hat{p}_i < \theta} (\text{Amount}_i + C_{\text{chargeback}}) + \sum_{i: y_i=0, \hat{p}_i \ge \theta} (C_{\text{friction}} + C_{\text{review}}) + \sum_{i: y_i=1, \hat{p}_i \ge \theta} C_{\text{review}}$$

The optimal operational decision threshold $\theta^*$ minimizes cumulative financial loss:
$$\theta^* = \arg\min_{\theta \in (0, 1)} \text{Cost}(\theta)$$


In [ ]:
from IPython.display import display
import os
import json
import warnings
import time
import concurrent.futures
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize_scalar

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    classification_report
)

try:
    from xgboost import XGBClassifier
    has_xgb = True
except ImportError:
    has_xgb = False

try:
    from lightgbm import LGBMClassifier
    has_lgb = True
except ImportError:
    has_lgb = False

try:
    from catboost import CatBoostClassifier
    has_cat = True
except ImportError:
    has_cat = False

plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 11
sns.set_palette('deep')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

def resolve_path(rel_path):
    candidates = [
        rel_path,
        os.path.join('..', rel_path),
        os.path.join('../..', rel_path),
        os.path.join(os.getcwd(), rel_path),
        os.path.join(os.path.dirname(os.getcwd()), rel_path)
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    for p in candidates:
        parent = os.path.dirname(p)
        if parent and os.path.exists(parent):
            return p
    return rel_path

print("Multi-Model Ensemble Stacking & Cost Optimization Engine initialized successfully.")


---
## 1. Leak-Free Feature Store Ingestion & Exposure Stream Extraction
Ingesting the engineered train, validation, and test partitions along with raw transaction amounts for empirical financial utility calculation.


In [ ]:
train_csv_gz = resolve_path('data/processed/train_features.csv.gz')
val_csv_gz = resolve_path('data/processed/val_features.csv.gz')
test_csv_gz = resolve_path('data/processed/test_features.csv.gz')

if os.path.exists(train_csv_gz):
    train_df = pd.read_csv(train_csv_gz, compression='gzip')
    val_df = pd.read_csv(val_csv_gz, compression='gzip')
    test_df = pd.read_csv(test_csv_gz, compression='gzip')
else:
    raw_path = resolve_path('data/raw/creditcard.parquet')
    if not os.path.exists(raw_path):
        raw_path = resolve_path('data/raw/creditcard.csv')
    raw_df = pd.read_parquet(raw_path) if raw_path.endswith('.parquet') else pd.read_csv(raw_path)
    raw_df = raw_df.sort_values(by='Time').reset_index(drop=True)
    n = len(raw_df)
    train_df = raw_df.iloc[:int(n*0.70)].copy()
    val_df = raw_df.iloc[int(n*0.70):int(n*0.85)].copy()
    test_df = raw_df.iloc[int(n*0.85):].copy()

raw_file = resolve_path('data/raw/creditcard.parquet')
if not os.path.exists(raw_file):
    raw_file = resolve_path('data/raw/creditcard.csv')
raw_all = pd.read_parquet(raw_file) if raw_file.endswith('.parquet') else pd.read_csv(raw_file)
raw_all = raw_all.sort_values(by='Time').reset_index(drop=True)
test_raw_amounts = raw_all.iloc[int(len(raw_all)*0.85):]['Amount'].values
val_raw_amounts = raw_all.iloc[int(len(raw_all)*0.70):int(len(raw_all)*0.85)]['Amount'].values

feature_cols = [c for c in train_df.columns if c != 'Class']

X_train, y_train = train_df[feature_cols].values, train_df['Class'].values
X_val, y_val = val_df[feature_cols].values, val_df['Class'].values
X_test, y_test = test_df[feature_cols].values, test_df['Class'].values

print(f"Train Feature Matrix:       {X_train.shape} (Frauds: {np.sum(y_train == 1):,})")
print(f"Validation Feature Matrix:  {X_val.shape} (Frauds: {np.sum(y_val == 1):,})")
print(f"Test Feature Matrix:        {X_test.shape} (Frauds: {np.sum(y_test == 1):,})")
print(f"Validation Fraud Exposure:  ${np.sum(val_raw_amounts[y_val == 1]):,.2f}")
print(f"OOT Test Fraud Exposure:    ${np.sum(test_raw_amounts[y_test == 1]):,.2f}")


---
## 2. Level-0 Heterogeneous Base Learner Portfolio Configuration
Configuring an ensemble portfolio of 7 diverse classification algorithms covering gradient boosting, oblivious decision trees, randomized forests, deep neural representations, and calibrated linear baselines:

1. **LightGBM Classifier**: Leaf-wise tree growth with gradient-based one-side sampling (`is_unbalance=True`).
2. **XGBoost Classifier**: Exact greedy split with second-order Taylor gradient expansion (`scale_pos_weight`).
3. **CatBoost Classifier**: Ordered boosting with symmetric oblivious decision trees.
4. **Balanced Random Forest**: Bootstrap aggregation with balanced subsample weighting.
5. **Balanced Extra Trees**: Extremely randomized tree splits for variance suppression.
6. **Histogram Gradient Boosting (HistGB)**: Fast binned gradient boosting.
7. **Balanced Logistic Regression**: $L_2$-regularized calibrated linear baseline.


In [ ]:
pos_weight = float(np.sum(y_train == 0) / max(1, np.sum(y_train == 1)))

base_models = {}

if has_lgb:
    base_models['LightGBM'] = LGBMClassifier(
        n_estimators=150,
        learning_rate=0.05,
        num_leaves=31,
        is_unbalance=True,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

if has_xgb:
    base_models['XGBoost'] = XGBClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=5,
        scale_pos_weight=pos_weight,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    )

if has_cat:
    base_models['CatBoost'] = CatBoostClassifier(
        iterations=150,
        learning_rate=0.05,
        depth=5,
        auto_class_weights='Balanced',
        random_seed=42,
        verbose=0,
        thread_count=-1
    )

base_models['RandomForest'] = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight='balanced_subsample',
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

base_models['ExtraTrees'] = ExtraTreesClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight='balanced',
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

base_models['HistGradientBoosting'] = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.05,
    max_depth=6,
    class_weight='balanced',
    random_state=42
)

base_models['LogisticRegression'] = LogisticRegression(
    C=0.1,
    class_weight='balanced',
    max_iter=1000,
    solver='lbfgs',
    random_state=42
)

print(f"Configured {len(base_models)} Level-0 Base Learners:")
for name in base_models:
    print(f" - {name}")


---
## 3. Stratified K-Fold Out-of-Fold (OOF) Prediction Generation
Generating leak-free Level-1 meta-features $\mathbf{Z}_{\text{OOF}} \in \mathbb{R}^{N_{\text{train}} \times M}$ and Out-of-Time test meta-features $\mathbf{Z}_{\text{test}} \in \mathbb{R}^{N_{\text{test}} \times M}$ using 5-Fold Stratified Cross-Validation.


In [ ]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_train_preds = {name: np.zeros(len(X_train)) for name in base_models}
val_preds = {name: np.zeros(len(X_val)) for name in base_models}
test_preds = {name: np.zeros(len(X_test)) for name in base_models}
base_metrics = []

for name, model_template in base_models.items():
    start_time = time.time()
    val_fold_preds = np.zeros(len(X_val))
    test_fold_preds = np.zeros(len(X_test))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr_f, y_tr_f = X_train[train_idx], y_train[train_idx]
        X_va_f, y_va_f = X_train[val_idx], y_train[val_idx]
        
        from sklearn.base import clone
        clf = clone(model_template)
        clf.fit(X_tr_f, y_tr_f)
        
        oof_train_preds[name][val_idx] = clf.predict_proba(X_va_f)[:, 1]
        val_fold_preds += clf.predict_proba(X_val)[:, 1] / n_splits
        test_fold_preds += clf.predict_proba(X_test)[:, 1] / n_splits
    
    val_preds[name] = val_fold_preds
    test_preds[name] = test_fold_preds
    elapsed = time.time() - start_time
    
    oof_prauc = average_precision_score(y_train, oof_train_preds[name])
    oof_roc = roc_auc_score(y_train, oof_train_preds[name])
    test_prauc = average_precision_score(y_test, test_preds[name])
    test_roc = roc_auc_score(y_test, test_preds[name])
    test_brier = brier_score_loss(y_test, test_preds[name])
    
    k_alerts = max(1, int(len(y_test) * 0.01))
    top_idx = np.argsort(test_preds[name])[::-1][:k_alerts]
    rec_at_1 = (np.sum(y_test[top_idx] == 1) / max(1, np.sum(y_test == 1))) * 100
    
    base_metrics.append({
        'Base Model': name,
        'OOF PR-AUC': oof_prauc,
        'OOF ROC-AUC': oof_roc,
        'OOT Test PR-AUC': test_prauc,
        'OOT Test ROC-AUC': test_roc,
        'Test Brier Loss': test_brier,
        'Recall @ Top 1%': rec_at_1,
        'CV Latency (s)': elapsed
    })
    print(f"Trained: {name:<22} | OOF PR-AUC: {oof_prauc:.4f} | OOT Test PR-AUC: {test_prauc:.4f} | Recall@1%: {rec_at_1:.2f}% | Latency: {elapsed:.2f}s")

df_base_results = pd.DataFrame(base_metrics).sort_values(by='OOT Test PR-AUC', ascending=False).reset_index(drop=True)
display(df_base_results)


---
## 4. Base Learner Diversity Forensics & Disagreement Diagnostics
Evaluating ensemble diversity metrics to verify that the Level-0 classifiers produce uncorrelated error residuals:
- **Prediction Correlation Matrix ($r_{p,q}$)**: Pairwise Pearson correlation between predicted fraud probabilities.
- **Disagreement Measure ($D_{p,q}$)**: Proportion of test instances where two base models assign discordant binary classifications.


In [ ]:
model_names = list(base_models.keys())
oof_matrix = np.column_stack([oof_train_preds[m] for m in model_names])
val_matrix = np.column_stack([val_preds[m] for m in model_names])
test_matrix = np.column_stack([test_preds[m] for m in model_names])

df_oof_corr = pd.DataFrame(oof_matrix, columns=model_names).corr()

disagreement_matrix = np.zeros((len(model_names), len(model_names)))
for i, m1 in enumerate(model_names):
    for j, m2 in enumerate(model_names):
        pred_i = (test_preds[m1] >= 0.5).astype(int)
        pred_j = (test_preds[m2] >= 0.5).astype(int)
        disagreement_matrix[i, j] = np.mean(pred_i != pred_j)

df_disagree = pd.DataFrame(disagreement_matrix, index=model_names, columns=model_names)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(df_oof_corr, annot=True, fmt='.3f', cmap='viridis', ax=axes[0], cbar_kws={'label': 'Pearson Correlation'})
axes[0].set_title('Level-0 Out-of-Fold (OOF) Prediction Correlation Matrix', fontweight='bold')

sns.heatmap(df_disagree * 100, annot=True, fmt='.2f', cmap='mako', ax=axes[1], cbar_kws={'label': 'Disagreement Rate (%)'})
axes[1].set_title('Pairwise Classification Disagreement Rate (%) on Test Partition', fontweight='bold')

plt.tight_layout()
plt.show()

print("Base Classifier Correlation Summary:")
display(df_oof_corr.describe())


---
## 5. Level-1 Meta-Learner Exploration & Stacking Optimization
Benchmarking 4 competing stacking meta-learner paradigms trained on $\mathbf{Z}_{\text{OOF}}$:

1. **Stacking Meta-Learner (Logistic Regression)**: $L_2$-regularized linear combination of logit-transformed base probabilities.
2. **Constrained Non-Negative Meta-Learner**: Optimization under non-negative weights $w_m \ge 0$ with simplex normalization $\sum w_m = 1$.
3. **Non-Linear Tree Meta-Learner (LightGBM Meta)**: Shallow gradient boosted tree (`max_depth=3`) to model non-linear base prediction interactions.
4. **Optimal Soft Voting / Rank-Averaging Blend**: Simplex weight optimization maximizing validation PR-AUC.


In [ ]:
meta_models = {}
meta_test_preds = {}
meta_val_preds = {}
meta_eval_results = []

meta_lr = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000, random_state=42)
meta_lr.fit(oof_matrix, y_train)
meta_models['Stacking Logistic Regression'] = meta_lr
meta_val_preds['Stacking Logistic Regression'] = meta_lr.predict_proba(val_matrix)[:, 1]
meta_test_preds['Stacking Logistic Regression'] = meta_lr.predict_proba(test_matrix)[:, 1]

meta_lgb = LGBMClassifier(
    n_estimators=50,
    max_depth=3,
    num_leaves=7,
    learning_rate=0.05,
    is_unbalance=True,
    random_state=42,
    verbose=-1
)
meta_lgb.fit(oof_matrix, y_train)
meta_models['Stacking LightGBM Meta'] = meta_lgb
meta_val_preds['Stacking LightGBM Meta'] = meta_lgb.predict_proba(val_matrix)[:, 1]
meta_test_preds['Stacking LightGBM Meta'] = meta_lgb.predict_proba(test_matrix)[:, 1]

from scipy.optimize import minimize
def loss_func(weights):
    weights = np.array(weights)
    weights = np.maximum(0, weights)
    s = np.sum(weights)
    if s == 0:
        return 1.0
    w_norm = weights / s
    blend_val = np.dot(val_matrix, w_norm)
    return -average_precision_score(y_val, blend_val)

init_weights = np.ones(len(model_names)) / len(model_names)
bounds = [(0, 1) for _ in range(len(model_names))]
opt_res = minimize(loss_func, init_weights, method='SLSQP', bounds=bounds)
opt_weights = opt_res.x / np.sum(opt_res.x)

meta_val_preds['Optimal Soft Voting Blend'] = np.dot(val_matrix, opt_weights)
meta_test_preds['Optimal Soft Voting Blend'] = np.dot(test_matrix, opt_weights)

equal_weights = np.ones(len(model_names)) / len(model_names)
meta_val_preds['Uniform Soft Voting Baseline'] = np.dot(val_matrix, equal_weights)
meta_test_preds['Uniform Soft Voting Baseline'] = np.dot(test_matrix, equal_weights)

best_single_name = df_base_results.iloc[0]['Base Model']
meta_val_preds[f'Single Best ({best_single_name})'] = val_preds[best_single_name]
meta_test_preds[f'Single Best ({best_single_name})'] = test_preds[best_single_name]

for name, preds in meta_test_preds.items():
    prauc = average_precision_score(y_test, preds)
    roc = roc_auc_score(y_test, preds)
    brier = brier_score_loss(y_test, preds)
    k_alerts = max(1, int(len(y_test) * 0.01))
    top_idx = np.argsort(preds)[::-1][:k_alerts]
    rec_at_1 = (np.sum(y_test[top_idx] == 1) / max(1, np.sum(y_test == 1))) * 100
    
    meta_eval_results.append({
        'Ensemble Strategy': name,
        'OOT PR-AUC (Average Precision)': prauc,
        'OOT ROC-AUC': roc,
        'Brier Score Loss': brier,
        'Recall @ Top 1% Alerts': rec_at_1
    })

df_meta_results = pd.DataFrame(meta_eval_results).sort_values(by='OOT PR-AUC (Average Precision)', ascending=False).reset_index(drop=True)
display(df_meta_results)


---
## 6. Financial Cost-Optimal Decision Threshold Optimization Engine
Evaluating financial risk exposure across decision cutoff thresholds $\theta \in [0.001, 0.999]$.

#### Asymmetric Financial Cost Parameters:
- **Chargeback Loss ($C_{\text{chargeback}}$)**: $\$25.00$ dispute penalty per missed fraud.
- **Direct Asset Exposure ($C_{\text{loss}}$)**: The stolen transaction dollar amount $\text{Amount}_i$.
- **Manual Review Investigation ($C_{\text{review}}$)**: $\$15.00$ operational cost per alert.
- **Customer Friction & Insult ($C_{\text{friction}}$)**: $\$50.00$ penalty per false positive alert due to friction and churn risk.

Total financial loss:
$$\text{Loss}(\theta) = \sum_{i: y_i=1, \hat{p}_i < \theta} (\text{Amount}_i + 25.0) + \sum_{i: y_i=0, \hat{p}_i \ge \theta} (50.0 + 15.0) + \sum_{i: y_i=1, \hat{p}_i \ge \theta} 15.0$$


In [ ]:
c_chargeback = 25.0
c_review = 15.0
c_friction = 50.0

def compute_financial_loss(y_true, y_probs, amounts, threshold):
    preds = (y_probs >= threshold).astype(int)
    fn_mask = (y_true == 1) & (preds == 0)
    fp_mask = (y_true == 0) & (preds == 1)
    tp_mask = (y_true == 1) & (preds == 1)
    
    fn_cost = np.sum(amounts[fn_mask] + c_chargeback)
    fp_cost = np.sum(fp_mask) * (c_friction + c_review)
    tp_cost = np.sum(tp_mask) * c_review
    
    total_cost = fn_cost + fp_cost + tp_cost
    return total_cost, fn_cost, fp_cost, tp_cost

threshold_grid = np.linspace(0.005, 0.995, 200)

primary_ensemble_name = 'Stacking Logistic Regression'
ensemble_val_probs = meta_val_preds[primary_ensemble_name]
ensemble_test_probs = meta_test_preds[primary_ensemble_name]
best_single_probs = meta_test_preds[f'Single Best ({best_single_name})']

val_costs = []
for th in threshold_grid:
    cost, _, _, _ = compute_financial_loss(y_val, ensemble_val_probs, val_raw_amounts, th)
    val_costs.append(cost)

optimal_idx = np.argmin(val_costs)
optimal_threshold = threshold_grid[optimal_idx]

test_cost_optimal, fn_c_opt, fp_c_opt, tp_c_opt = compute_financial_loss(
    y_test, ensemble_test_probs, test_raw_amounts, optimal_threshold
)
test_cost_default, fn_c_def, fp_c_def, tp_c_def = compute_financial_loss(
    y_test, ensemble_test_probs, test_raw_amounts, 0.50
)
test_cost_single, fn_c_sng, fp_c_sng, tp_c_sng = compute_financial_loss(
    y_test, best_single_probs, test_raw_amounts, optimal_threshold
)
total_exposure = np.sum(test_raw_amounts[y_test == 1] + c_chargeback)

cost_comparison = [
    {
        'Operational Policy': 'No Model (100% Uncaught Exposure)',
        'Decision Threshold': 'N/A',
        'Total Dollar Loss ($)': total_exposure,
        'Dollar Savings vs No-Model ($)': 0.0,
        'Cost Reduction (%)': 0.0
    },
    {
        'Operational Policy': 'Ensemble Stacking (Default Threshold)',
        'Decision Threshold': 0.5000,
        'Total Dollar Loss ($)': test_cost_default,
        'Dollar Savings vs No-Model ($)': total_exposure - test_cost_default,
        'Cost Reduction (%)': ((total_exposure - test_cost_default) / total_exposure) * 100
    },
    {
        'Operational Policy': f'Single Best Model ({best_single_name} @ Optimal)',
        'Decision Threshold': optimal_threshold,
        'Total Dollar Loss ($)': test_cost_single,
        'Dollar Savings vs No-Model ($)': total_exposure - test_cost_single,
        'Cost Reduction (%)': ((total_exposure - test_cost_single) / total_exposure) * 100
    },
    {
        'Operational Policy': 'Ensemble Stacking @ Cost-Optimal Threshold',
        'Decision Threshold': optimal_threshold,
        'Total Dollar Loss ($)': test_cost_optimal,
        'Dollar Savings vs No-Model ($)': total_exposure - test_cost_optimal,
        'Cost Reduction (%)': ((total_exposure - test_cost_optimal) / total_exposure) * 100
    }
]

df_cost_comp = pd.DataFrame(cost_comparison)
display(df_cost_comp)
print(f"Optimal Decision Threshold (Theta*): {optimal_threshold:.4f}")
print(f"Ensemble Loss at Theta*:            ${test_cost_optimal:,.2f}")
print(f"Dollar Savings over Default 0.5:    ${test_cost_default - test_cost_optimal:,.2f}")


---
## 7. Operational Risk Curves & Threshold Sensitivity Diagnostics
Visualizing the financial risk optimization landscape:
1. **Financial Loss vs Decision Threshold Curve**: Demonstrating global cost minimization at $\theta^*$.
2. **Precision-Recall & ROC Discrimination Frontier**: Comparing base models against the ensemble meta-learner.
3. **Alert Volume vs Fraud Recall Profile**: Quantifying fraud capture capacity under restricted analyst capacity (Top 0.5%, 1.0%, 2.0% alert rates).
4. **Optimal Decision Confusion Matrix & Error Cost Breakdown**.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

test_loss_curve = []
for th in threshold_grid:
    loss, _, _, _ = compute_financial_loss(y_test, ensemble_test_probs, test_raw_amounts, th)
    test_loss_curve.append(loss)

axes[0, 0].plot(threshold_grid, test_loss_curve, color='navy', lw=2.5, label='Ensemble Stacking Loss')
axes[0, 0].axvline(optimal_threshold, color='crimson', linestyle='--', lw=2, label=f'Optimal Theta* ({optimal_threshold:.4f})')
axes[0, 0].axvline(0.50, color='gray', linestyle=':', lw=1.5, label='Default 0.50 Threshold')
axes[0, 0].scatter([optimal_threshold], [test_cost_optimal], color='crimson', s=100, zorder=5)
axes[0, 0].set_title('Financial Loss ($) vs Decision Threshold Cutoff', fontweight='bold')
axes[0, 0].set_xlabel('Decision Threshold (Theta)')
axes[0, 0].set_ylabel('Total Financial Dollar Loss ($)')
axes[0, 0].legend(loc='upper right')
axes[0, 0].grid(True, alpha=0.3)

for name in ['Stacking Logistic Regression', 'Optimal Soft Voting Blend', f'Single Best ({best_single_name})', 'Uniform Soft Voting Baseline']:
    p, r, _ = precision_recall_curve(y_test, meta_test_preds[name])
    score = average_precision_score(y_test, meta_test_preds[name])
    axes[0, 1].plot(r, p, lw=2, label=f'{name} (PR-AUC: {score:.4f})')

axes[0, 1].set_title('Out-of-Time Test Precision-Recall Frontier', fontweight='bold')
axes[0, 1].set_xlabel('Recall (Fraud Detection Rate)')
axes[0, 1].set_ylabel('Precision (Positive Predictive Value)')
axes[0, 1].legend(loc='lower left')
axes[0, 1].grid(True, alpha=0.3)

alert_fractions = np.linspace(0.001, 0.05, 50)
for name in ['Stacking Logistic Regression', 'Optimal Soft Voting Blend', f'Single Best ({best_single_name})']:
    recalls = []
    for frac in alert_fractions:
        k = max(1, int(len(y_test) * frac))
        top_idx = np.argsort(meta_test_preds[name])[::-1][:k]
        rec = (np.sum(y_test[top_idx] == 1) / max(1, np.sum(y_test == 1))) * 100
        recalls.append(rec)
    axes[1, 0].plot(alert_fractions * 100, recalls, lw=2.2, label=name)

axes[1, 0].set_title('Fraud Recall (%) vs Alert Queue Budget (% of Traffic)', fontweight='bold')
axes[1, 0].set_xlabel('Alert Queue Capacity (% of Total Transactions)')
axes[1, 0].set_ylabel('Fraud Recall Captured (%)')
axes[1, 0].axvline(1.0, color='crimson', linestyle='--', alpha=0.7, label='Standard 1% SLA Budget')
axes[1, 0].legend(loc='lower right')
axes[1, 0].grid(True, alpha=0.3)

opt_preds = (ensemble_test_probs >= optimal_threshold).astype(int)
cm = confusion_matrix(y_test, opt_preds)
cm_labels = np.array([
    [f"TN: {cm[0,0]:,}\nClean Passed", f"FP: {cm[0,1]:,}\nCustomer Friction"],
    [f"FN: {cm[1,0]:,}\nMissed Fraud Loss", f"TP: {cm[1,1]:,}\nFraud Intercepted"]
])

sns.heatmap(cm, annot=cm_labels, fmt='', cmap='Blues', cbar=False, ax=axes[1, 1],
            xticklabels=['Predicted Legitimate', 'Predicted Fraud'],
            yticklabels=['Actual Legitimate', 'Actual Fraud'])
axes[1, 1].set_title(f'Optimal Confusion Matrix (Theta = {optimal_threshold:.4f})', fontweight='bold')

plt.tight_layout()
plt.show()


---
## 8. Real-Time Authorization SLA & Inference Latency Profiling
Measuring microsecond latency distributions for online scoring:
- **Sequential Base Model Inference**: Cumulative latency of evaluating base models sequentially.
- **Parallel Multi-Threaded Inference**: Concurrent execution via `ThreadPoolExecutor` for high-throughput gateway SLAs.


In [ ]:
sample_batch = X_test[:1000]

fitted_base_models = {}
for name, template in base_models.items():
    from sklearn.base import clone
    m = clone(template)
    m.fit(X_train, y_train)
    fitted_base_models[name] = m

seq_latencies = []
for _ in range(100):
    t0 = time.perf_counter()
    b_preds = [fitted_base_models[m].predict_proba(sample_batch[:1])[:, 1] for m in fitted_base_models]
    meta_in = np.column_stack(b_preds)
    final_score = meta_lr.predict_proba(meta_in)[:, 1]
    seq_latencies.append((time.perf_counter() - t0) * 1000)

def predict_single_model(m_name, x):
    return fitted_base_models[m_name].predict_proba(x)[:, 1]

par_latencies = []
with concurrent.futures.ThreadPoolExecutor(max_workers=len(fitted_base_models)) as executor:
    for _ in range(100):
        t0 = time.perf_counter()
        futures = [executor.submit(predict_single_model, m, sample_batch[:1]) for m in fitted_base_models]
        b_preds = [f.result() for f in futures]
        meta_in = np.column_stack(b_preds)
        final_score = meta_lr.predict_proba(meta_in)[:, 1]
        par_latencies.append((time.perf_counter() - t0) * 1000)

seq_p50, seq_p95, seq_p99 = np.percentile(seq_latencies, [50, 95, 99])
par_p50, par_p95, par_p99 = np.percentile(par_latencies, [50, 95, 99])

latency_summary = [
    {
        'Inference Execution Mode': 'Sequential Base Scoring',
        'p50 Latency (ms)': seq_p50,
        'p95 Latency (ms)': seq_p95,
        'p99 Latency (ms)': seq_p99,
        'SLA Threshold (<15ms)': 'Compliant' if seq_p99 < 15.0 else 'Warning'
    },
    {
        'Inference Execution Mode': 'Parallel Multi-Threaded Scoring',
        'p50 Latency (ms)': par_p50,
        'p95 Latency (ms)': par_p95,
        'p99 Latency (ms)': par_p99,
        'SLA Threshold (<15ms)': 'Compliant' if par_p99 < 15.0 else 'Warning'
    }
]

df_latency = pd.DataFrame(latency_summary)
display(df_latency)


---
## 9. Production Stacking Manifest & Parameter Serialization
Exporting the complete ensemble specification, base model weights, optimal threshold $\theta^*$, and governance metrics to `data/ensemble_stacking_manifest.json`.


In [ ]:
manifest = {
    'platform': 'Enterprise Financial Risk Intelligence & Fraud Forensics',
    'notebook': '12_Ensemble_Stacking_and_Cost_Optimal_Thresholding.ipynb',
    'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'level_0_base_models': list(base_models.keys()),
    'meta_learner': 'Stacking Logistic Regression (L2 Regularized)',
    'optimal_decision_threshold': float(optimal_threshold),
    'cost_parameters': {
        'chargeback_loss_fee': c_chargeback,
        'analyst_review_cost': c_review,
        'customer_friction_insult_penalty': c_friction
    },
    'performance_benchmarks': {
        'ensemble_oot_prauc': float(average_precision_score(y_test, ensemble_test_probs)),
        'ensemble_oot_rocauc': float(roc_auc_score(y_test, ensemble_test_probs)),
        'ensemble_brier_score': float(brier_score_loss(y_test, ensemble_test_probs)),
        'optimal_policy_total_loss': float(test_cost_optimal),
        'default_policy_total_loss': float(test_cost_default),
        'uncaught_exposure_total_loss': float(total_exposure),
        'dollar_savings_over_default': float(test_cost_default - test_cost_optimal)
    },
    'inference_latency_sla': {
        'p50_ms': float(seq_p50),
        'p95_ms': float(seq_p95),
        'p99_ms': float(seq_p99)
    }
}

manifest_path = resolve_path('data/ensemble_stacking_manifest.json')
parent_dir = os.path.dirname(manifest_path)
if parent_dir:
    os.makedirs(parent_dir, exist_ok=True)

with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print(f"Production Ensemble Stacking Manifest exported successfully to: {manifest_path}")
print("Manifest Summary:")
print(json.dumps(manifest['performance_benchmarks'], indent=2))
